# NVIDIA Dynamo Qwen3-32B: single-node and multi-node tensor parallelism

This notebook serves [Qwen/Qwen3-32B](https://huggingface.co/Qwen/Qwen3-32B) with
[NVIDIA Dynamo](https://github.com/ai-dynamo/dynamo) four different ways on the same hardware
budget of 8 L40S GPUs, and shows how to prove which one you actually got:

| Step | Values file | Topology |
| --- | --- | --- |
| 3 | [`dgd-singlenode-tp4.yaml`](dgd-singlenode-tp4.yaml) | `TP=4` in one pod on one node |
| 5 | [`dgd-multinode-tp8.yaml`](dgd-multinode-tp8.yaml) | `TP=8` across two nodes over EFA, via LeaderWorkerSet |
| 7 | [`dgd-multinode-tp4-pp2.yaml`](dgd-multinode-tp4-pp2.yaml) | `TP=4` within each node, `PP=2` across the boundary |
| 8 | [`dgd-router-kv-2x.yaml`](dgd-router-kv-2x.yaml) | two single-node replicas behind a KV-aware router |

**Read [README.md](README.md) before running this.** Qwen3-32B *fits on one node*, so the
multi-node steps here are deliberately not the fastest way to serve this model — they are the
cheapest way to learn what a node boundary costs before you depend on one. The README also covers
six prerequisites that do not fail loudly, four of which have to be satisfied by Terraform rather
than by a values file.

There is no container to build: the runtime image is pulled anonymously from `nvcr.io`. Qwen3-32B
is not gated, so no Hugging Face token is required.

## Setup and Imports

In [ ]:
! pip install kubernetes
! pip install openai

In [ ]:
import os
import subprocess
import sys

# Set working directory
os.chdir(os.path.expanduser('~/amazon-eks-machine-learning-with-terraform-and-kubeflow'))
print(f"Working directory: {os.getcwd()}")

# Get the src directory
src_dir = os.path.join(os.getcwd(), "src")
sys.path.insert(0, src_dir)

from k8s.utils import (
    wait_for_helm_release_pods,
    wait_for_dynamo_ready,
    find_k8s_service
)

# Get notebook directory
notebook_dir = os.path.join(os.getcwd(), 'examples', 'inference', 'dynamo', 'vllm', 'qwen3-32b')
print(f"Notebook directory: {notebook_dir}")

# initialize key variables
release_name = 'dyn-qwen3-32b'
namespace = 'kubeflow-user-example-com'
hf_model_id = 'Qwen/Qwen3-32B'
served_model_name = 'qwen3-32b'

## Step 1: Verify the Platform

Three things have to be running before any of this works, and they come from three different
places. The Dynamo operator and Volcano are installed by Terraform when `dynamo_enabled = true`;
the LeaderWorkerSet controller is installed by `main.tf` unconditionally.

**Volcano is a hard gate on the multi-node steps, and it fails silently.** The Dynamo operator
refuses to author a `LeaderWorkerSet` without it: with Volcano absent the `DynamoGraphDeployment`
is accepted, the operator logs nothing useful, and no pods are ever created. It is operator
policy rather than an LWS requirement — see [README.md](README.md#1-volcano-installed-with-dynamo_enabled).

In [ ]:
for ns in ['dynamo-system', 'lws-system', 'volcano-system']:
    cmd = ['kubectl', 'get', 'pods', '-n', ns]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"===== {ns} =====")
    print(result.stdout or result.stderr)

# The LWS CRD is what the operator needs CRUD on for the multi-node steps
cmd = ['kubectl', 'get', 'crd', 'leaderworkersets.leaderworkerset.x-k8s.io']
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout or result.stderr)

## Step 2: Verify the EFA Prerequisites

Skip this and the multi-node steps still "work" — at TCP speed, or not at all, with nothing in
any log to say why. Two independent settings in `main.tf` have to admit g6e, and **satisfying one
without the other is the failure mode to look for**: a `cudaefa` NodePool that admits g6e without
a device plugin that advertises on it gives you a node that launches, never publishes
`vpc.amazonaws.com/efa`, and leaves the pod `Pending` on a resource nothing provides.

The cell below checks the NodePool. If `g6e.12xlarge` is absent from the instance-type
requirement, stop here and fix Terraform — see
[README.md](README.md#2-the-cudaefa-nodepool-must-admit-your-instance-type).

In [ ]:
import json

cmd = ['kubectl', 'get', 'nodepool', 'cudaefa', '-o', 'json']
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print("cudaefa NodePool not found:", result.stderr)
else:
    np = json.loads(result.stdout)
    for req in np['spec']['template']['spec']['requirements']:
        if req['key'] == 'node.kubernetes.io/instance-type':
            types = req['values']
            print(f"cudaefa admits {len(types)} instance types:")
            print("  " + ", ".join(sorted(types)))
            print()
            print("g6e.12xlarge admitted:", 'g6e.12xlarge' in types)

    # Karpenter resolves this pool to the subnets tagged from cuda_efa_az, a single AZ. EFA
    # traffic is not routed between subnets, so every pod in a group must land in that zone.
    cmd = ['kubectl', 'get', 'ec2nodeclass', 'cudaefa', '-o',
           'jsonpath={.status.subnets[*].zone}']
    result = subprocess.run(cmd, capture_output=True, text=True)
    print("\ncudaefa resolves to zone(s):", result.stdout or "(none resolved yet)")

Already-running GPU nodes are the other half of the check. A node in the `cudaefa` pool should
report **both** `nvidia.com/gpu` and `vpc.amazonaws.com/efa` as allocatable. A `4` with a blank
next to it means the device plugin is not advertising, which is
[prerequisite 3](README.md#3-the-efa-device-plugin-must-advertise-on-them).

In [ ]:
cmd = ['kubectl', 'get', 'nodes', '-l', 'karpenter.sh/nodepool=cudaefa', '-o',
       'custom-columns=NODE:.metadata.name,TYPE:.metadata.labels.node\\.kubernetes\\.io/instance-type,'
       'ZONE:.metadata.labels.topology\\.kubernetes\\.io/zone,'
       'GPU:.status.allocatable.nvidia\\.com/gpu,EFA:.status.allocatable.vpc\\.amazonaws\\.com/efa']
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout or "No cudaefa nodes yet -- Karpenter provisions them in Step 3.")

## Step 3: Download the Model Weights

Qwen3-32B is not gated, so no Hugging Face token is needed. The weights land at
`/fsx/pretrained-models/Qwen/Qwen3-32B` — about 65 GB in bf16 — which is the path every
values file in this directory reads.

This needs no GPU, so it can run while the cluster is still building.

In [ ]:
cmd = [
    'helm', 'install', '--debug', f'{release_name}-weights',
    'charts/machine-learning/model-prep/hf-snapshot',
    '--set-json', f'env=[{{"name":"HF_MODEL_ID","value":"{hf_model_id}"}}]',
    '-n', namespace
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
# wait for model download job to complete
wait_for_helm_release_pods(release_name=f'{release_name}-weights', namespace=namespace)

In [ ]:
# Uninstall the Helm chart after completion
cmd = ['helm', 'uninstall', f'{release_name}-weights', '-n', namespace]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

## Step 4: Single-Node TP=4 — the Baseline to Beat

[`dgd-singlenode-tp4.yaml`](dgd-singlenode-tp4.yaml) runs one worker at `TP=4` on one
`g6e.12xlarge`. All four tensor-parallel ranks talk over the local bus, so no fabric is involved
and nothing here depends on EFA.

**Run this arm first and keep its behaviour in mind.** Every multi-node arm below is slower, and
the point of the exercise is to find out by how much — a boundary you can measure is a boundary
you can decide about.

In [ ]:
cmd = [
    'helm', 'install', '--debug', release_name,
    'charts/machine-learning/serving/dynamo/',
    '-f', f'{notebook_dir}/dgd-singlenode-tp4.yaml',
    '-n', namespace
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
# Provisioning a 4-GPU node and loading ~65 GB off FSx takes a while on a cold filesystem
wait_for_dynamo_ready(release_name=release_name, namespace=namespace, timeout=2400)

### Confirm this really is single-node

The check that matters is that **no LeaderWorkerSet exists**. The operator only injects the
multi-node launch path when `TP x PP` exceeds the container's GPU request; at `TP=4` with 4 GPUs
it does not, so this should be an ordinary Deployment with one worker pod.

That asymmetry is worth internalising now, because it is also the trap in Step 5: a values file
that asks for `TP=4` on a `node_count: 2` component builds a single-node engine on the leader and
leaves the second node idle, with no error.

In [ ]:
cmd = ['kubectl', 'get', 'lws', '-n', namespace]
result = subprocess.run(cmd, capture_output=True, text=True)
print("LeaderWorkerSets:", result.stdout.strip() or result.stderr.strip())

cmd = ['kubectl', 'get', 'pods', '-n', namespace, '-o',
       'custom-columns=POD:.metadata.name,NODE:.spec.nodeName,'
       'GPU:.spec.containers[0].resources.requests.nvidia\\.com/gpu']
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

# vLLM states the parallel layout it actually built, which beats inferring it from the flags
cmd = ['kubectl', 'logs', '-n', namespace, '-l', 'nvidia.com/dynamo-component-type=worker',
       '--tail=2000']
result = subprocess.run(cmd, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if any(k in line for k in ('world_size', 'tensor_model_parallel', 'Available KV cache memory',
                               'GPU KV cache size')):
        print(line)

### Test the Service with the OpenAI API

This assumes a `kubectl port-forward` to the frontend on `localhost:8000`:

    kubectl port-forward -n kubeflow-user-example-com svc/dyn-qwen3-32b-frontend 8000:8000

First confirm a worker has registered its model. An empty list means the frontend is up but no
worker has registered — read the worker pod's log, not the frontend's.

In [ ]:
import time
from openai import OpenAI, APIConnectionError

client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

def list_models(client, attempts=30, interval=10):
    """List served models, waiting for the port-forward rather than failing on a refused
    connection. A `kubectl port-forward` is bound to one pod, so it dies whenever the frontend
    pod is replaced -- which makes a dead tunnel look exactly like a broken deployment."""
    for attempt in range(attempts):
        try:
            return [model.id for model in client.models.list()]
        except APIConnectionError:
            print(f"No answer on localhost:8000 (attempt {attempt + 1}/{attempts}). "
                  f"Is the port-forward running?")
            time.sleep(interval)
    raise RuntimeError("Gave up waiting for the frontend on localhost:8000")

print(list_models(client))

In [ ]:
response = client.chat.completions.create(
    model=served_model_name,
    messages=[{"role": "user", "content": "Explain the concept of attention masking in causal language models and its importance for autoregressive generation."}],
    max_tokens=2048
)

print(response.choices[0].message.content)

Qwen3 is a reasoning model and emits a `<think>` block before its answer, so keep `max_tokens`
generous. At 128 the response is truncated mid-reasoning and looks like a broken deployment rather
than a budget you set too low.

## Step 5: Multi-Node TP=8 Across Two Nodes

Uninstall first. This is not optional: the admission webhook rejects any change to the *set of
component names* on an existing `DynamoGraphDeployment` with `component topology is immutable and
cannot be modified after creation`, and even for a permitted change the operator hashes the
component spec into the `DynamoComponentDeployment` name, so an upgrade strands the previous
generation on its GPUs.

In [ ]:
cmd = ['helm', 'uninstall', release_name, '-n', namespace]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

### What you are installing, and what you are *not* writing

**You do not write a LeaderWorkerSet.** `node_count: 2` is the entire multi-node input:

```yaml
VllmWorker:
  type: worker
  node_count: 2
```

The chart renders that to `spec.components[].multinode.nodeCount` and the operator authors the LWS
itself — one per multi-node component, `size: nodeCount`, `restartPolicy:
RecreateGroupOnPodRestart`, separate leader and worker pod templates. LWS then injects
`LWS_LEADER_ADDRESS`, `LWS_WORKER_INDEX` and `LWS_GROUP_SIZE` into every pod in the group. Stable
identity is the whole reason an LWS is used instead of a Deployment: rank 0 has to be findable by
name before any process has started.

This is the opposite of the
[`triton-inference-server-lws`](../../../../../charts/machine-learning/serving/triton-inference-server-lws/)
chart in this repository, which owns the `LeaderWorkerSet` object directly and ships a
`launcher.sh` to start a Ray head. Nothing like that exists here.

Two things in [`dgd-multinode-tp8.yaml`](dgd-multinode-tp8.yaml) are load-bearing in ways that are
invisible if you skip them:

- **`--disable-custom-all-reduce`** — mandatory on any GPU without NVLink, which is every g6e.
  Without it all 8 ranks hang in vLLM's distributed init and the engine never loads a weight. vLLM
  *detects* the situation, logs `Custom collectives are disabled because this multi-node group does
  not support MNNVL multicast`, and then calls `_init_mnnvl_buffer()` anyway, which blocks forever
  in `torch.distributed._symmetric_memory.rendezvous()`. It presents as a slow model load, not as
  a crash: the pod stays `Running`, nothing is logged after that line, and there is no timeout.
- **`node_selector: karpenter.sh/nodepool: cudaefa`** — pinning the instance type is not enough.
  The `cuda` pool's EC2NodeClass selects subnets in every AZ, so without this the two pods can land
  in different zones and the fabric is silently dead, falling back to TCP with no error anywhere.

In [ ]:
cmd = [
    'helm', 'install', '--debug', release_name,
    'charts/machine-learning/serving/dynamo/',
    '-f', f'{notebook_dir}/dgd-multinode-tp8.yaml',
    '-n', namespace
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
# Two nodes to provision and 8 ranks to initialize, so allow longer than the single-node arm.
wait_for_dynamo_ready(release_name=release_name, namespace=namespace, timeout=3600)

### Confirm the Two Nodes Are Really One Engine

"The pods are Running" proves nothing here. Four separate things can each leave you with a
deployment that looks healthy and is not what you asked for: no LWS at all, an LWS whose pods
share a node, a group split across availability zones with a dead fabric, or an engine that built
`TP=4` on the leader and ignored the second node.

The cell below checks all four. `LWS_GROUP_SIZE` and the vLLM `world_size` are the two that
cannot be faked.

In [ ]:
cmd = ['kubectl', 'get', 'lws', '-n', namespace, '-o', 'wide']
result = subprocess.run(cmd, capture_output=True, text=True)
print("===== LeaderWorkerSet =====")
print(result.stdout or "NONE -- the operator did not author one. Check Volcano is installed.")

# One pod per node, in the same zone, each holding 4 GPUs and 1 EFA device
cmd = ['kubectl', 'get', 'pods', '-n', namespace,
       '-l', 'nvidia.com/dynamo-component-type=worker', '-o',
       'custom-columns=POD:.metadata.name,ROLE:.metadata.labels.leaderworkerset\\.sigs\\.k8s\\.io/worker-index,'
       'NODE:.spec.nodeName,GPU:.spec.containers[0].resources.requests.nvidia\\.com/gpu,'
       'EFA:.spec.containers[0].resources.requests.vpc\\.amazonaws\\.com/efa']
result = subprocess.run(cmd, capture_output=True, text=True)
print("===== worker pods =====")
print(result.stdout)

# Both pods must be in the SAME zone: EFA traffic is not routed between subnets
cmd = ['kubectl', 'get', 'pods', '-n', namespace,
       '-l', 'nvidia.com/dynamo-component-type=worker',
       '-o', 'jsonpath={range .items[*]}{.spec.nodeName}{"\n"}{end}']
nodes = subprocess.run(cmd, capture_output=True, text=True).stdout.split()
zones = set()
for node in nodes:
    cmd = ['kubectl', 'get', 'node', node, '-o',
           'jsonpath={.metadata.labels.topology\\.kubernetes\\.io/zone}']
    zones.add(subprocess.run(cmd, capture_output=True, text=True).stdout.strip())
print(f"===== placement =====\n{len(set(nodes))} distinct nodes, zone(s): {zones}")
if len(zones) > 1:
    print("WARNING: group is split across zones. EFA is not routed between subnets -- this "
          "deployment is silently running over TCP.")

### How vLLM Actually Spans the Nodes: `mp`, not Ray

This is the part most likely to surprise anyone who has built multi-node vLLM by hand, where the
usual answer is a Ray cluster. Dynamo 1.5.0 uses vLLM's **native multiprocessing** backend: the
leader runs the engine and the workers join over `torch.distributed`. No Ray head, no etcd, no
NATS, no MPI.

The values file sets `nvidia.com/vllm-distributed-executor-backend: "mp"` explicitly rather than
inheriting it, because unset, the operator picks between `mp` and Ray from a version gate — which
means the same values file could launch two different topologies on two different clusters.

The cell below reads it back off the running engine, and confirms NCCL selected the EFA provider
rather than falling back to sockets.

In [ ]:
cmd = ['kubectl', 'logs', '-n', namespace, '-l', 'nvidia.com/dynamo-component-type=worker',
       '--tail=4000', '--prefix']
result = subprocess.run(cmd, capture_output=True, text=True)

interesting = ('world_size', 'distributed_executor_backend', 'LWS_', 'Loading safetensors',
               'Available KV cache memory', 'GPU KV cache size', 'graph capturing',
               'MNNVL', 'NCCL version', 'Using network', 'isAllDirectP2p')
for line in result.stdout.splitlines():
    if any(k in line for k in interesting):
        print(line)

#### If the load appears to hang

A multi-node engine that never becomes ready has two very different causes that look identical
from the outside, and telling them apart saves an hour of waiting on something that will never
finish:

| | MNNVL deadlock | slow cold FSx read |
| --- | --- | --- |
| `Loading safetensors checkpoint shards` line | never appears | appears, advances slowly |
| `rchar` in `/proc/<rank-pid>/io`, sampled 30s apart | flat | climbing at tens of MB/s |
| `nvidia-smi` on the leader | 0%, ~800 MiB | 0%, climbing |
| `py-spy dump --pid <rank>` | `rendezvous` / `_init_mnnvl_buffer` | a weight-loading frame |

The second column is the normal case on a filesystem that has not read these weights before: FSx
imports from S3 lazily, so the first launch pays for the fetch. The cell below samples `rchar`,
which is the cheapest of the four to check.

In [ ]:
import time

cmd = ['kubectl', 'get', 'pods', '-n', namespace,
       '-l', 'nvidia.com/dynamo-component-type=worker',
       '-o', 'jsonpath={.items[0].metadata.name}']
leader = subprocess.run(cmd, capture_output=True, text=True).stdout.strip()

def total_rchar(pod):
    """Sum bytes read by every python process in the pod. Climbing means weights are loading."""
    script = ('for p in /proc/[0-9]*/io; do grep -H ^rchar $p 2>/dev/null; done '
              '| awk -F: \'{s+=$3} END {print s}\'')
    out = subprocess.run(['kubectl', 'exec', '-n', namespace, pod, '--', 'bash', '-c', script],
                         capture_output=True, text=True).stdout.strip()
    return int(out) if out.isdigit() else 0

a = total_rchar(leader)
time.sleep(30)
b = total_rchar(leader)
print(f"{leader}: rchar {a/1e9:.1f} GB -> {b/1e9:.1f} GB over 30s "
      f"({(b - a)/30/1e6:.0f} MB/s)")
print("Flat and no safetensors line above => MNNVL deadlock, not a slow load.")

### Test the Multi-Node Service

The API surface is unchanged, which is the point — the node boundary is a serving decision, not a
client-visible one.

**Restart your `kubectl port-forward` first.** The reinstall replaced the frontend pod, and a
port-forward is bound to the pod it started against, so the old tunnel is dead.

In [ ]:
client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")
print(list_models(client))

response = client.chat.completions.create(
    model=served_model_name,
    messages=[{"role": "user", "content": "Explain the concept of attention masking in causal language models and its importance for autoregressive generation."}],
    max_tokens=2048
)

print(response.choices[0].message.content)

### The Negative Control

[`dgd-multinode-tp8-tcp.yaml`](dgd-multinode-tp8-tcp.yaml) is the same deployment with the fabric
taken away — no EFA device request, and libfabric left to fall back to sockets. Run it if you want
to know what the interconnect is buying you, because a multi-node arm that is accidentally running
over TCP looks exactly like one that is working.

Comparing against it is the only way to be sure the EFA request did something. It is the same
install command with a different `-f`.

## Step 6: TP=4 Within Each Node, PP=2 Across the Boundary

[`dgd-multinode-tp4-pp2.yaml`](dgd-multinode-tp4-pp2.yaml) keeps the same 8 GPUs and the same two
nodes, and changes only *which* collective crosses the network.

This is the interesting comparison in the whole notebook. `TP=8` puts a tensor-parallel all-reduce
on the network for every layer; `TP=4` + `PP=2` keeps the all-reduces on each node's local bus and
sends only pipeline activations across the boundary, which is far less traffic. The cost is
pipeline bubbles at low concurrency.

In [ ]:
cmd = ['helm', 'uninstall', release_name, '-n', namespace]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

cmd = [
    'helm', 'install', '--debug', release_name,
    'charts/machine-learning/serving/dynamo/',
    '-f', f'{notebook_dir}/dgd-multinode-tp4-pp2.yaml',
    '-n', namespace
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
wait_for_dynamo_ready(release_name=release_name, namespace=namespace, timeout=3600)

# Confirm the engine built the layout you asked for rather than silently collapsing it
cmd = ['kubectl', 'logs', '-n', namespace, '-l', 'nvidia.com/dynamo-component-type=worker',
       '--tail=4000']
result = subprocess.run(cmd, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if any(k in line for k in ('world_size', 'pipeline_parallel', 'tensor_model_parallel',
                               'GPU KV cache size')):
        print(line)

## Step 7: The Same Hardware Spent on Replicas Instead — KV-Aware Routing

[`dgd-router-kv-2x.yaml`](dgd-router-kv-2x.yaml) spends the same 8 GPUs on **two independent
single-node `TP=4` replicas** behind a KV-aware router. No node boundary inside an engine, no
Volcano gang scheduling, no LWS. It scales with replica count rather than engine size, and cannot
serve a model larger than one node holds — which makes it the right default for this model and
the wrong one for Step 5's.

Routing is where this arm earns its keep, and it is also where a benchmark will most easily lie to
you. **Five things have to line up and not one of them fails loudly.** `--router-mode kv` is
accepted without complaint on a single-worker deployment, with prefix caching off, with workers
that publish nothing, and on a workload that shares no prefixes — behaving like round-robin and
reporting nothing wrong in every one of those cases:

1. **The frontend runs a KV router** — `--router-mode kv` on `dynamo.frontend`. Default is
   `round-robin`.
2. **The workers publish cache state, which needs its own flag.** This is the one that catches
   people: `--router-mode kv` on the frontend does *not* make the workers feed it. Without
   `--kv-events-config '{"enable_kv_cache_events": true, "publisher": "zmq"}'` on the worker, the
   frontend's radix tree stays empty for the lifetime of the deployment and the worker says so
   exactly once at startup, with `use_kv_events=False`. `publisher` has to be named too — it
   defaults to `None`, which publishes nothing.
3. **The frontend maintains a radix tree** of which blocks live on which worker, built from those
   events.
4. **Prefix caching must be on in the engine**, or the routing decision is correct and worthless.
   On by default in vLLM V1; the values file sets `--enable-prefix-caching` explicitly so a future
   default change breaks loudly instead of flattening the result to zero.
5. **The workload has to share prefixes**, on more distinct prefixes than you have workers, and
   its prefix sequence must not be phase-locked to the router. That is a property of your traffic,
   not your configuration.

In [ ]:
cmd = ['helm', 'uninstall', release_name, '-n', namespace]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

cmd = [
    'helm', 'install', '--debug', release_name,
    'charts/machine-learning/serving/dynamo/',
    '-f', f'{notebook_dir}/dgd-router-kv-2x.yaml',
    '-n', namespace
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

wait_for_dynamo_ready(release_name=release_name, namespace=namespace, timeout=3600)

### Proving the Router Did Something

There are **two instruments and you need both**, because they measure different things and can
disagree:

- `bench/router-metrics.sh` reads `dynamo_component_router_*` on the frontend — what the radix
  tree believed, and how much overlap it thought it was buying. These metrics **exist only in
  `--router-mode kv`**; in round-robin every one is flat zero, so they cannot compare a mode
  against its own control.
- `bench/engine-cache.sh` reads `vllm:prefix_cache_hits_total` and
  `vllm:prefix_cache_queries_total` from each worker's own Prometheus registry, which the Dynamo
  worker serves on **port 9090**, not the frontend's 8000. These are engine-side and therefore
  reported identically in every routing mode, which is the only reason a kv-vs-round-robin
  comparison is possible at all. Counters are cumulative since worker start, so take a delta
  around the run.

The gap between the two is the whole point: a router can report a healthy mean overlap while
round-robin reaches exactly the same engine hit rate. Only the engine counters can tell you the
router was working perfectly and buying nothing. No latency table ever will.

Matching happens at **KV block granularity**, not per token — 16 tokens per block here, reported
as `dynamo_frontend_model_kv_cache_block_size`. A 790-token shared prefix matches 49 whole blocks
and the remainder is recomputed, so a perfect hit reads as ~0.98, not 1.0.

In [ ]:
bench = os.path.join(os.getcwd(), 'examples', 'inference', 'dynamo', 'vllm', 'qwen3-8b', 'bench')
env = {**os.environ, 'NS': namespace,
       'FRONTEND': f'{release_name}-frontend',
       'WORKER': f'{release_name}-vllmworker'}

for script in ['router-metrics.sh', 'engine-cache.sh']:
    print(f"===== {script} =====")
    result = subprocess.run(['bash', os.path.join(bench, script)],
                            capture_output=True, text=True, env=env)
    print(result.stdout or result.stderr)

Read the **overlap histogram**, `dynamo_component_router_kv_hit_rate`. A working KV router
produces a bimodal distribution — a spike near 0 for first-time prefixes and a spike near 1 for
repeats. A mean overlap of 0 with a nonzero decision count means the mechanism is live and your
workload has nothing for it to find. The four states worth distinguishing:

| `routing decisions` | `KV event batches` | mean overlap | engine hit rate vs control | diagnosis |
| --- | --- | --- | --- | --- |
| 0 | 0 | n/a | — | frontend is not in `kv` mode at all |
| > 0 | 0 | 0.0000 | — | `kv` mode, but workers are not publishing — add `--kv-events-config` |
| > 0 | > 0 | 0.0000 | same | live, but the workload shares no prefixes |
| > 0 | > 0 | high | same | router working, buying nothing — the workload is phase-locked or has fewer prefixes than workers |
| > 0 | > 0 | high | better | working as intended |

To generate traffic with shared prefixes, [`bench/loadgen.py`](../qwen3-8b/bench/loadgen.py)
groups prompts behind a common prefix. It must run **inside** the cluster — a `kubectl
port-forward` tunnel is a single userspace hop on your workstation and saturates long before the
GPUs do, flattening exactly the differences you are trying to see.

In [ ]:
# One arm's worth of load. Run the same command against a round-robin control -- a hit rate
# with nothing to compare it against is not a result.
bench_sh = os.path.join(bench, 'run-bench.sh')
result = subprocess.run(
    ['bash', bench_sh, 'router-kv-a',
     '--prompt-tokens', '1000', '--max-tokens', '128',
     '--concurrency', '1,4,16', '--requests', '32'],
    capture_output=True, text=True, env={**env, 'LOG': '/tmp/router-kv-a.log'})
print(result.stdout[-4000:] or result.stderr[-4000:])

## Step 8: Stop the Service

Karpenter's `consolidationPolicy` decides what happens to the GPU nodes once the pods are gone.
Check it before assuming they will be reclaimed — with `consolidateAfter: Never` the empty nodes
stay, which is what you want mid-experiment and not what you want afterwards.

In [ ]:
cmd = ['helm', 'uninstall', release_name, '-n', namespace]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

cmd = ['kubectl', 'get', 'nodepool', '-o',
       'custom-columns=NAME:.metadata.name,POLICY:.spec.disruption.consolidationPolicy,'
       'AFTER:.spec.disruption.consolidateAfter']
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

## Where to Go Next

- [README.md](README.md) — the full write-up: the six prerequisites, how the operator generates
  the LWS, the three different things called "KV cache something", and the findings from each arm.
- [`../qwen3-235b-a22b-fp8/serve.ipynb`](../qwen3-235b-a22b-fp8/serve.ipynb) — the case where
  multi-node is not a comparison but a requirement: a model that does not fit on one node at any
  `--gpu-memory-utilization`.
- [`../qwen3-8b/serve.ipynb`](../qwen3-8b/serve.ipynb) — aggregated vs disaggregated prefill and
  decode, which is an orthogonal axis to everything here.